In [ ]:
import requests
import csv
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "T2M,WS2M,RH2M,PRECTOT,CLOUD_AMT",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20230101,
    "end": 20231231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()["properties"]["parameter"]

temp = data["T2M"]
wind = data["WS2M"]
humidity = data["RH2M"]
rain = data["PRECTOTCORR"]
cloud = data["CLOUD_AMT"]

with open("data/powercut.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "date","temperature",
        "humidity","wind_speed","rainfall",
        "cloud","power_cut"
    ])

    for date in temp:

        t = temp[date]
        w = wind[date]
        h = humidity[date]
        r = rain[date]
        c = cloud[date]

        # 🔥 improved rule
        if (r > 8 and c > 80) or (w > 10) or (h > 90 and c > 85):
            power_cut = 1
        else:
            power_cut = 0

        writer.writerow([date, t, h, w, r, c, power_cut])

print("⚠️ Power cut dataset created!")

⚠️ Power cut dataset created!


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

df = pd.read_csv("data/powercut.csv")

# feature engineering
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

features = [
    "month",
    "temperature",
    "humidity",
    "wind_speed",
    "rainfall",
    "cloud"
]

X = df[features]
y = df["power_cut"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# model
model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

# evaluation
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("\n⚠️ POWER CUT MODEL PERFORMANCE")
print("Accuracy:", round(acc*100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

joblib.dump(model, "models/powercut.pkl")

print("\n✅ Power cut model trained!")


⚠️ POWER CUT MODEL PERFORMANCE
Accuracy: 94.52 %

Confusion Matrix:
 [[63  1]
 [ 3  6]]

✅ Power cut model trained!


In [ ]:
import joblib
import pandas as pd

model = joblib.load("models/powercut.pkl")

# -----------------------------
# INPUT
# -----------------------------
input_data = {
    "month": 6,
    "temperature": 28,
    "humidity": 92,
    "wind_speed": 12,
    "rainfall": 15,
    "cloud": 90
}   

df = pd.DataFrame([input_data])

# -----------------------------
# PREDICT PROBABILITY
# -----------------------------
proba = model.predict_proba(df)[0]

# class 0 = no power cut
# class 1 = power cut
prob_no_cut = proba[0]
prob_cut = proba[1]

# -----------------------------
# DECISION
# -----------------------------
if prob_cut > 0.5:
    status = "⚠️ POWER CUT LIKELY"
else:
    status = "✅ NORMAL GRID"

# -----------------------------
# OUTPUT
# -----------------------------
print(f"\n🔹 Probability of Power Cut: {round(prob_cut, 3)}")
print(f"🔹 Probability of Normal   : {round(prob_no_cut, 3)}")
print(f"\n{status}")


🔹 Probability of Power Cut: 0.91
🔹 Probability of Normal   : 0.09

⚠️ POWER CUT LIKELY


In [11]:
import joblib
import pandas as pd

model = joblib.load("models/powercut.pkl")

tests = [
    ("☀️ Normal Weather", 2, 3, 30),
    ("🌧️ Heavy Rain", 15, 5, 90),
    ("🌪️ Strong Wind", 0, 12, 60),
    ("⛈️ Storm", 20, 15, 95)
]

for name, rain, wind, cloud in tests:

    df = pd.DataFrame([{
        "month": 6,
        "temperature": 28,
        "humidity": 90,
        "wind_speed": wind,
        "rainfall": rain,
        "cloud": cloud
    }])

    pred = model.predict(df)[0]
    proba = model.predict_proba(df)[0]

    print(f"\n{name}")
    print("Result:", "⚠️ Power Cut" if pred else "✅ Normal", proba)


☀️ Normal Weather
Result: ✅ Normal [0.665 0.335]

🌧️ Heavy Rain
Result: ⚠️ Power Cut [0.09 0.91]

🌪️ Strong Wind
Result: ✅ Normal [0.66 0.34]

⛈️ Storm
Result: ⚠️ Power Cut [0.09 0.91]


In [14]:
import pandas as pd

df = pd.read_csv("data/powercut.csv")
df.describe()

,date,temperature,humidity,wind_speed,rainfall,cloud,power_cut
count,3.650000e+02,365.000000,365.000000,365.000000,365.000000,365.000000,365.000000
mean,2.023067e+07,28.004110,72.097945,2.393425,2.902658,63.283644,0.112329
std,3.454755e+02,2.692262,9.636482,0.663810,5.204531,30.203139,0.316204
min,2.023010e+07,21.700000,47.720000,0.890000,0.000000,0.730000,0.000000
25%,2.023040e+07,25.920000,65.250000,1.930000,0.050000,40.470000,0.000000
50%,2.023070e+07,28.260000,72.190000,2.320000,0.670000,71.840000,0.000000
75%,2.023100e+07,29.980000,78.740000,2.780000,3.720000,89.650000,0.000000
max,2.023123e+07,33.660000,92.390000,4.570000,53.120000,99.980000,1.000000
